# 02 — Train the MRZ line recognizer

One cell. Set `REPO`, pick a GPU pod, run.

**Run `01_synthetic_preview.ipynb` first.** Everything this model learns comes from the
generator; if the samples look wrong, the model will be wrong and this notebook will not
tell you.

## What this trains

A **PARSeq-lineage ViT encoder with a purpose-built fixed-length decoder** — not stock
PARSeq. The deviation is deliberate and worth understanding before you spend GPU hours:

| | stock PARSeq | here | why |
| --- | --- | --- | --- |
| input | 32×128 (4:1) | 32×704 (22:1) | 44 chars in 128px is 2.9px each — unreadable |
| encoder | ViT-Small, 12 layers | ViT-tiny, 6 layers | ViT-S measured **75.5ms/line** on CPU; two lines alone would blow the 100ms budget |
| decoder | autoregressive, 25 steps | one shot, 44 positions | MRZ has no language prior to learn, and Phase 4 wants per-position marginals |
| charset | 36 lowercase | 37 (`A-Z0-9<`) | the MRZ alphabet |
| loss | CE | CE + label smoothing | CTC solves unknown alignment and length; TD3 has neither |

**Output contract:** `(batch, 44, 37)` log-probs. Phase 4's beam search and ICAO validation
read exactly this.

## Expected cost

~38k steps at batch 128. On an A10/A100, roughly 1–3 hours. The generator produces ~1250
samples/s across 8 workers — if `it/s × batch_size` approaches that, the GPU is waiting on
the CPU and you should raise `num_workers` or lower `dpi`.

In [ ]:
# ============================================================================
# Train the MRZ recognizer. Self-contained.
# ============================================================================
import subprocess, sys, os, pathlib

REPO = "https://github.com/Kamisadev/mrz-ai.git"
WORKDIR = pathlib.Path("/workspace") if pathlib.Path("/workspace").exists() else pathlib.Path.cwd()
PROJECT = WORKDIR / "mrz_ai_v2"

def sh(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

if not PROJECT.exists():
    PROJECT = pathlib.Path.cwd() if pathlib.Path.cwd().joinpath("src/mrz_ai").exists() else PROJECT
    if not PROJECT.exists():
        sh("git", "clone", "--depth", "1", REPO, PROJECT)

import importlib.util
REQUIRED = {"PIL": "pillow", "numpy": "numpy", "cv2": "opencv-python-headless",
            "torch": "torch", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if not importlib.util.find_spec(mod)]
if missing:
    sh(sys.executable, "-m", "pip", "install", "-q", *missing)

if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))

import torch
from mrz_ai.training.recognition import TrainConfig, Stage, train_recognition

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

# Checkpoints go to the persistent volume so a stopped pod does not lose them.
OUTPUT = (WORKDIR / "checkpoints/recognition") if WORKDIR.name == "workspace" \
         else PROJECT / "checkpoints/recognition"

config = TrainConfig(
    batch_size=128,
    learning_rate=7e-4,
    num_workers=min(os.cpu_count() or 4, 12),
    output_dir=OUTPUT,
    # The blueprint's curriculum as a severity sweep. Each stage keeps the
    # easier range below it, so the model does not forget clean documents.
    curriculum=(
        Stage("clean",    (0.0, 0.05),  2_000),
        Stage("light",    (0.0, 0.35),  6_000),
        Stage("moderate", (0.0, 0.65), 10_000),
        Stage("heavy",    (0.0, 1.0),  20_000),
    ),
)

checkpoint = train_recognition(config)
print("checkpoint:", checkpoint)

## Reading the numbers

`char` is per-character accuracy; `line` is the fraction of lines correct in *all 44*
positions. Watch `line` — one wrong character is a wrong document, so per-character
accuracy flatters the model badly here. At 44 characters, even 99.5% per character is only
~80% of lines correct.

Both are measured at full severity (0–1) on a disjoint generator seed, so there is no
leakage. Note what that does *not* mean: it is still synthetic evaluating synthetic. A
line accuracy of 99% here is a claim about this generator, not about passports.

Phase 4 will lift the effective accuracy above whatever this reports, because ICAO check
digits can reject a wrong hypothesis and promote the runner-up — but only for the fields
check digits cover. Sex and nationality have no protection at all (`docs/parser.md`).